In [37]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import os

In [79]:
raw_ihs_poverty_2010 = pd.read_csv(r'd:\GG\source\householdpoverty_10.CSV')
raw_ihs_weight_2010 = pd.read_csv(r'd:\GG\source\householdweight_10.CSV')
raw_ihs_weight_2015 = pd.read_csv(r'd:\GG\source\householdweight_15.CSV')
raw_ihs_poverty_2015 = pd.read_csv(r'd:\GG\source\householdpoverty_15.CSV')
raw_dhs_2020 = pd.read_stata(r'd:\GG\source\household_19_20.DTA')
raw_findex_2021 = pd.read_csv(r'd:\GG\source\connectivity_21.csv')
raw_findex_2024 = pd.read_csv(r'd:\GG\source\connectivity_24.csv')

In [89]:
# aggregate economic rank
# 2010
df_2010 = pd.DataFrame()
df_2010['hid'] = raw_ihs_poverty_2010['hid']
df_2010['survey_year'] = 2010
df_2010['data_source'] = 'IHS'
df_2010['lga'] = raw_ihs_poverty_2010['lga']

weight_lookup = raw_ihs_weight_2010[['hid', 'weightslga']].drop_duplicates(subset=['hid'])
df_2010 = df_2010.merge(weight_lookup, on='hid', how='left')
df_2010['weightslga'] = df_2010['weightslga'].fillna(np.nan)

poverty_map = raw_ihs_poverty_2010.drop_duplicates('hid').set_index('hid')['s11q2']
df_2010['hh_income'] = df_2010['hid'].astype(int).map(poverty_map)

# 2015
df_2015 = pd.DataFrame()
df_2015['hh_id'] = raw_ihs_poverty_2015['hid']
df_2015['survey_year'] = 2015
df_2015['data_source'] = 'IHS'
df_2015['eanum'] = raw_ihs_poverty_2015['eanum']

weight_lookup = raw_ihs_weight_2015[['eanum', 'hhweight']].drop_duplicates(subset=['eanum'])
df_2015 = df_2015.merge(weight_lookup, on='eanum', how='left')
df_2015['hh_weight'] = df_2015['hhweight'].fillna(np.nan)

poverty_map = raw_ihs_poverty_2015.drop_duplicates('hid').set_index('hid')['s13q3']
df_2015['hh_income'] = df_2015['hh_id'].astype(int).map(poverty_map)

# 2020
df_2020 = pd.DataFrame()
df_2020['hh_id'] = raw_dhs_2020['hhid']
df_2020['survey_year'] = raw_dhs_2020['hv007']
df_2020['hh_income'] = raw_dhs_2020['hv270']

df_2020['data_source'] = 'DHS'
df_2020['weight'] = raw_dhs_2020['hv005']

quintile_labels = {'poorest': 1, 'poorer': 2, 'middle': 3, 'richer': 4, 'richest': 5}
df_2020['hh_income'] = df_2020['hh_income'].map(quintile_labels)

# 2021
df_2021 = pd.DataFrame()
df_2021['hh_id'] = raw_findex_2021.index.map(lambda x: f"findex_21_{x}")
df_2021['survey_year'] = 2021
df_2021['data_source'] = 'Findex'
df_2021['hh_income'] = raw_findex_2021['inc_q']
df_2021['weight'] = (raw_findex_2021['wgt']).fillna(NA)

# 2024
df_2024 = pd.DataFrame()
df_2024['hh_id'] = raw_findex_2024.index.map(lambda x: f"findex_24_{x}")
df_2024['survey_year'] = 2024
df_2024['data_source'] = 'Findex'
df_2024['hh_income'] = raw_findex_2024['inc_q']
df_2024['weight'] = (raw_findex_2024['wgt']).fillna(NA)

In [91]:
# econ rank construct
df_2010['hh_econ_rank'] = (df_2010.sort_values('hh_income')['weightslga'].cumsum() - 0.5 * df_2010['weightslga']) / df_2010['weightslga'].sum() * 100
df_2015['hh_econ_rank'] = (df_2015.sort_values('hh_income')['hh_weight'].cumsum() - 0.5 * df_2015['hh_weight']) / df_2015['hh_weight'].sum() * 100
df_2020['hh_econ_rank'] = (df_2020.sort_values('hh_income')['weight'].cumsum() - 0.5 * df_2020['weight']) / df_2020['weight'].sum() * 100
df_2021['hh_econ_rank'] = (df_2021.sort_values('hh_income')['weight'].cumsum() - 0.5 * df_2021['weight']) / df_2021['weight'].sum() * 100
df_2024['hh_econ_rank'] = (df_2024.sort_values('hh_income')['weight'].cumsum() - 0.5 * df_2024['weight']) / df_2024['weight'].sum() * 100

In [99]:
df_2010

,hh_id,survey_year,data_source,lga,weight,hh_income,hh_econ_rank
0,1101101000110003101,2010,IHS,1,0.59,2.0,51.423247
1,1101101000110003102,2010,IHS,1,0.59,1.0,0.620941
2,1101101000110003103,2010,IHS,1,0.59,2.0,7.880949
3,1101101000110003104,2010,IHS,1,0.59,1.0,7.723077
4,1101101000110003105,2010,IHS,1,0.59,2.0,7.893270
...,...,...,...,...,...,...,...
4776,8802838302983024210,2010,IHS,8,0.98,2.0,8.282832
4777,8802838304983024201,2010,IHS,8,0.98,2.0,50.022762
4778,8802838304983024202,2010,IHS,8,0.98,2.0,50.067242
4779,8802838304983024203,2010,IHS,8,0.98,2.0,50.231796


In [100]:
df_2015

,hh_id,survey_year,data_source,eanum,hhweight,weight,hh_income,hh_econ_rank
0,1010101,2015,IHS,10101,18.111139,18.111139,3,53.797766
1,1010103,2015,IHS,10101,18.111139,18.111139,2,10.816436
2,1010104,2015,IHS,10101,18.111139,18.111139,2,10.822805
3,1010105,2015,IHS,10101,18.111139,18.111139,2,10.829174
4,1010106,2015,IHS,10101,18.111139,18.111139,1,0.042149
...,...,...,...,...,...,...,...,...
13276,8622216,2015,IHS,86222,10.126095,10.126095,3,53.792801
13277,8622217,2015,IHS,86222,10.126095,10.126095,2,10.909941
13278,8622218,2015,IHS,86222,10.126095,10.126095,2,53.711474
13279,8622219,2015,IHS,86222,10.126095,10.126095,1,10.784278


In [101]:
df_2020

,hh_id,survey_year,hh_income,data_source,weight,hh_econ_rank
0,1 1,2019,5,DHS,172113,99.998686
1,1 6,2019,3,DHS,172113,40.032339
2,1 10,2019,2,DHS,172113,33.796339
3,1 14,2019,3,DHS,172113,40.037889
4,1 18,2019,3,DHS,172113,40.047959
...,...,...,...,...,...,...
6544,281 36,2020,1,DHS,590524,17.749835
6545,281 38,2020,1,DHS,590524,17.764652
6546,281 39,2020,1,DHS,590524,17.782595
6547,281 41,2020,1,DHS,590524,4.519642


In [102]:
df_2021

,hh_id,survey_year,data_source,hh_income,weight,hh_econ_rank
0,findex_21_0,2021,Findex,4,0.496558,79.848035
1,findex_21_1,2021,Findex,4,1.878779,79.729268
2,findex_21_2,2021,Findex,3,0.220206,39.990227
3,findex_21_3,2021,Findex,4,0.939390,61.186075
4,findex_21_4,2021,Findex,2,0.504958,20.121523
...,...,...,...,...,...,...
995,findex_21_995,2021,Findex,2,2.215698,39.868432
996,findex_21_996,2021,Findex,5,1.351114,80.022958
997,findex_21_997,2021,Findex,1,0.496418,0.024821
998,findex_21_998,2021,Findex,5,0.373981,79.936704


In [103]:
df_2024

,hh_id,survey_year,data_source,hh_income,weight,hh_econ_rank
0,findex_24_0,2024,Findex,5,0.344393,99.754789
1,findex_24_1,2024,Findex,1,0.609212,19.221244
2,findex_24_2,2024,Findex,4,0.798520,60.921866
3,findex_24_3,2024,Findex,1,0.253243,19.264025
4,findex_24_4,2024,Findex,3,0.531983,40.868119
...,...,...,...,...,...,...
1003,findex_24_1003,2024,Findex,5,0.328288,81.159517
1004,findex_24_1004,2024,Findex,2,0.777492,20.392583
1005,findex_24_1005,2024,Findex,1,0.671328,19.898279
1006,findex_24_1006,2024,Findex,4,0.483620,79.927619


In [106]:
# concat all
df_2010.rename(columns={'hid': 'hh_id', 'weightslga': 'weight'}, inplace=True)
df_2015.rename(columns={'hh_weight': 'weight'}, inplace=True)

target_cols = ['hh_id', 'survey_year', 'data_source', 'hh_income', 'weight', 'hh_econ_rank']

df_2010 = df_2010[target_cols]
df_2015 = df_2015[target_cols]
df_2020 = df_2020[target_cols]
df_2021 = df_2021[target_cols]
df_2024 = df_2024[target_cols]

dfs = [df_2010, df_2015, df_2020, df_2021, df_2024]
for df in dfs:
    df['hh_id'] = df['hh_id'].astype(str)

df_panel = pd.concat(dfs, ignore_index=True)
df_panel = df_panel.dropna(subset=['hh_id', 'hh_income', 'weight'])

df_panel['unique_id'] = (
    df_panel['data_source'] + '_' + 
    df_panel['survey_year'].astype(str) + '_' + 
    df_panel['hh_id']
)

df_panel.set_index('unique_id', inplace=True)

display(df_panel.head())

,hh_id,survey_year,data_source,hh_income,weight,hh_econ_rank
unique_id,,,,,,
IHS_2010_1101101000110003101,1101101000110003101,2010,IHS,2.0,0.59,51.423247
IHS_2010_1101101000110003102,1101101000110003102,2010,IHS,1.0,0.59,0.620941
IHS_2010_1101101000110003103,1101101000110003103,2010,IHS,2.0,0.59,7.880949
IHS_2010_1101101000110003104,1101101000110003104,2010,IHS,1.0,0.59,7.723077
IHS_2010_1101101000110003105,1101101000110003105,2010,IHS,2.0,0.59,7.893270
